# LeetCode 646 — Maximum Length of Pair Chain bằng Python

Link đề: https://leetcode.com/problems/maximum-length-of-pair-chain/ · Độ khó: **Medium** · Chủ đề: Greedy, Array, Dynamic Programming, Sorting, Longest Increasing Subsequence

Skeleton LeetCode:

```python
class Solution:
    def findLongestChain(self, pairs: List[List[int]]) -> int:
        ...
```

Ghi chú: các ví dụ thêm ngoài hai ví dụ của đề (ví dụ 3 và 4 ở mục 2) do tôi tự đặt, đáp án đã kiểm chứng bằng vét cạn; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho `n` cặp số `[left, right]` (với `left < right`); cặp `[c, d]` **đi sau** cặp `[a, b]` nếu `b < c` (**nhỏ hơn hẳn**, không được bằng). Tìm **chuỗi dài nhất** các cặp liên tiếp thỏa điều kiện đó, được chọn bất kỳ cặp nào và theo thứ tự tùy ý.
2. Nắm các công cụ Python: `sorted(..., key=lambda p: p[1])`, so sánh/lấy phần tử của `list` lồng nhau, `max` với generator, `bisect_left`, `float("-inf")`, `itertools.combinations` (cho oracle) và mặt nạ bit để duyệt mọi tập con.
3. Đi từ **thử mọi cách chọn (đệ quy chọn/bỏ)** → **quy hoạch động O(n²) kiểu LIS sau khi sắp xếp** → **tham lam theo mút phải nhỏ nhất** → **tham lam theo mút trái, "co" mút phải khi chồng lấn** → **quy hoạch động + tìm kiếm nhị phân (`tails`) O(n log n)**.
4. Hiểu mấu chốt: bài này là **bài chọn nhiều khoảng không chồng nhau nhất** (activity selection); sắp theo **mút phải** thì luôn nên chọn cặp **kết thúc sớm nhất**, vì nó chừa nhiều chỗ nhất cho các cặp sau.
5. Tránh các lỗi kinh điển: dùng `<=` thay vì `<`, sắp theo mút trái rồi tham lam mà không co mút phải, khởi tạo mút phải hiện tại bằng `0` (hỏng với số âm), lấy `dp[-1]` thay vì `max(dp)`, và quên sắp xếp trước khi quy hoạch động.

## 2. Đề bài & ví dụ

Cho mảng `pairs` gồm `n` cặp, `pairs[i] = [left_i, right_i]` với `left_i < right_i`.

Cặp `p2 = [c, d]` **đi sau** cặp `p1 = [a, b]` nếu **`b < c`**. Một **chuỗi (chain)** các cặp được tạo theo cách này: mỗi cặp đi sau cặp ngay trước nó.

Hãy trả về **độ dài của chuỗi dài nhất** có thể tạo được. Không cần dùng hết các cặp, và có thể **chọn các cặp theo thứ tự bất kỳ** (nên thứ tự trong input không quan trọng).

**Ví dụ 1** (của đề)

- Input: `pairs = [[1,2],[2,3],[3,4]]`
- Output: `2`
- Giải thích: chuỗi dài nhất là `[1,2] -> [3,4]`. Lưu ý `[1,2] -> [2,3]` **không** hợp lệ vì `2 < 2` sai.

**Ví dụ 2** (của đề)

- Input: `pairs = [[1,2],[7,8],[4,5]]`
- Output: `3`
- Giải thích: `[1,2] -> [4,5] -> [7,8]`.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `pairs = [[1,100],[2,3],[4,5]]`
- Output: `2`
- Giải thích: chọn `[2,3] -> [4,5]`; cặp "to" `[1,100]` bao trùm cả hai nên chọn nó chỉ được độ dài `1`.

**Ví dụ 4 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `pairs = [[-10,-8],[8,9],[-5,0],[6,10],[-6,-4],[1,7],[9,10],[-4,7]]`
- Output: `4`
- Giải thích: ví dụ có số âm và nhiều cặp chồng lấn; chuỗi cụ thể được tìm ra bằng code ở mục 8 (cách tham lam in ra chuỗi tối ưu).

**Ràng buộc:**

- `n == pairs.length`, `1 <= n <= 1000`
- `-1000 <= left_i < right_i <= 1000`

In [1]:
from typing import List
from bisect import bisect_left
from itertools import combinations
import random
import time

examples = [
    ([[1, 2], [2, 3], [3, 4]], 2),
    ([[1, 2], [7, 8], [4, 5]], 3),
    ([[1, 100], [2, 3], [4, 5]], 2),
    ([[-10, -8], [8, 9], [-5, 0], [6, 10], [-6, -4], [1, 7], [9, 10], [-4, 7]], 4),
]
for pairs, expected in examples:
    print(pairs, "-> kỳ vọng", expected)

[[1, 2], [2, 3], [3, 4]] -> kỳ vọng 2
[[1, 2], [7, 8], [4, 5]] -> kỳ vọng 3
[[1, 100], [2, 3], [4, 5]] -> kỳ vọng 2
[[-10, -8], [8, 9], [-5, 0], [6, 10], [-6, -4], [1, 7], [9, 10], [-4, 7]] -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Danh sách lồng nhau và lấy hai đầu của một cặp

`pairs` là list của các list nhỏ `[a, b]`. `p[0]` là mút trái, `p[1]` là mút phải. Có thể **gỡ cặp ngay trong vòng `for`**: `for a, b in pairs:`.

In [2]:
pairs = [[1, 2], [7, 8], [4, 5]]
print(pairs[1], pairs[1][0], pairs[1][1])         # [7, 8] 7 8
for a, b in pairs:
    print(f"cặp ({a}, {b}) dài {b - a}")

[7, 8] 7 8
cặp (1, 2) dài 1
cặp (7, 8) dài 1
cặp (4, 5) dài 1


### 3.2. Điều kiện "đi sau": nhỏ hơn **hẳn**

`p2 = [c, d]` đi sau `p1 = [a, b]` khi **`b < c`**. Dấu `<` là **chặt**: hai cặp chạm nhau ở một điểm (`b == c`) thì **không** nối được. Ta đặt hàm nhỏ để dùng lại:

In [3]:
def follows(p1, p2):
    """p2 có đi sau p1 không?"""
    return p1[1] < p2[0]

print(follows([1, 2], [3, 4]))    # True  (2 < 3)
print(follows([1, 2], [2, 3]))    # False (2 < 2 sai)
print(follows([3, 4], [1, 2]))    # False (4 < 1 sai) -> thứ tự có quan trọng

True
False
False


### 3.3. Sắp xếp theo khóa: `sorted(key=...)`

`sorted(pairs, key=lambda p: p[1])` sắp các cặp theo **mút phải** tăng dần (không đổi `pairs` gốc). `lambda p: p[1]` là hàm nhỏ lấy khóa sắp xếp. Có thể sắp theo nhiều khóa bằng tuple: `key=lambda p: (p[1], p[0])` (phải trước, rồi tới trái). Ngoài ra `list.sort(key=...)` sắp **tại chỗ**.

In [4]:
pairs = [[3, 9], [1, 2], [4, 5], [0, 5]]
print(sorted(pairs))                                        # theo mút trái rồi mút phải (mặc định so sánh list)
print(sorted(pairs, key=lambda p: p[0]))                    # theo mút trái
print(sorted(pairs, key=lambda p: p[1]))                    # theo mút phải
print(sorted(pairs, key=lambda p: (p[1], p[0])))            # phải, hòa thì theo trái
print(pairs)                                                # không bị đổi

[[0, 5], [1, 2], [3, 9], [4, 5]]
[[0, 5], [1, 2], [3, 9], [4, 5]]
[[1, 2], [4, 5], [0, 5], [3, 9]]
[[1, 2], [0, 5], [4, 5], [3, 9]]
[[3, 9], [1, 2], [4, 5], [0, 5]]


### 3.4. `max` với generator và `default`

`max(iterable, default=...)` lấy phần tử lớn nhất; nếu iterable **rỗng** mà không có `default` sẽ lỗi `ValueError`. Rất tiện khi viết `dp[i] = 1 + max((dp[j] for j in range(i) if ...), default=0)`.

In [5]:
print(max([3, 1, 4]))
print(max((x for x in range(5) if x > 10), default=0))      # generator rỗng -> 0 nhờ default
try:
    max([])
except ValueError as e:
    print("ValueError:", e)

4
0
ValueError: max() iterable argument is empty


### 3.5. `float("-inf")` và vì sao không khởi tạo bằng `0`

Ta sẽ cần "mút phải của chuỗi hiện tại, khi chưa chọn cặp nào". Giá trị đó phải **nhỏ hơn mọi mút trái có thể**. Vì mút trái có thể **âm** (tới `-1000`), khởi tạo `0` là sai; dùng `float("-inf")` (âm vô cực) luôn đúng.

In [6]:
NEG = float("-inf")
print(NEG < -1000, NEG < 0, max(NEG, -5))      # True True -5
print("nếu khởi tạo 0, cặp [-5,-3] có đi sau 'chuỗi rỗng'? ", 0 < -5, "(sai!)")
print("nếu khởi tạo -inf:                                  ", NEG < -5, "(đúng)")

True True -5
nếu khởi tạo 0, cặp [-5,-3] có đi sau 'chuỗi rỗng'?  False (sai!)
nếu khởi tạo -inf:                                   True (đúng)


### 3.6. `bisect_left`: tìm vị trí trong list đã sắp

`bisect_left(a, x)` trả về chỉ số `i` nhỏ nhất sao cho `a[i] >= x` (với `a` sắp tăng). Tức là `a[:i]` đều `< x` và `a[i:]` đều `>= x`. Nó chạy `O(log n)`. Ta sẽ dùng để hỏi "có bao nhiêu phần tử **nhỏ hơn hẳn** `x`".

In [7]:
a = [2, 5, 5, 9]
print(bisect_left(a, 5), bisect_left(a, 6), bisect_left(a, 1), bisect_left(a, 10))   # 1 3 0 4
print("số phần tử < 5:", bisect_left(a, 5), "| số phần tử < 6:", bisect_left(a, 6))

1 3 0 4
số phần tử < 5: 1 | số phần tử < 6: 3


### 3.7. Nhìn bài như đồ thị / như khoảng

Mỗi cặp là một **khoảng** `[a, b]` trên trục số. Hai khoảng nối được khi chúng **không chung điểm nào** và cái sau nằm **bên phải hẳn**. Chuỗi dài nhất = **nhiều khoảng đôi một không giao nhau nhất** (vì không giao ⇒ có thể xếp theo thứ tự trái → phải thành một chuỗi). Đây chính là bài **chọn hoạt động (activity selection)** kinh điển.

Cũng có thể nhìn là bài **dãy con tăng dài nhất (LIS)**: sau khi sắp theo mút trái, ta tìm dãy con mà "mút phải của cặp trước `<` mút trái của cặp sau".

## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét `[[1,100],[2,3],[4,5],[6,7]]`. Hãy vẽ trục số:

```
 1 ------------------------------------------ 100    [1,100]
   2-3                                                 [2,3]
        4-5                                            [4,5]
             6-7                                       [6,7]
```

Cặp `[1,100]` bao trùm tất cả nên chọn nó **chặn** mọi cặp khác; ta thấy rằng nên chọn các cặp "ngắn và kết thúc sớm": `[2,3] -> [4,5] -> [6,7]` được `3`.

**Cách nghĩ tay (tham lam):** đi từ trái sang phải trên trục số; mỗi lần chọn cặp **kết thúc sớm nhất** trong số các cặp còn bắt đầu **sau** mút phải hiện tại.

1. Chưa chọn gì: cặp kết thúc sớm nhất là `[2,3]` (mút phải `3`). Chọn. `end = 3`.
2. Các cặp bắt đầu sau `3`: `[4,5]`, `[6,7]`. Kết thúc sớm nhất: `[4,5]`. Chọn. `end = 5`.
3. Còn `[6,7]` bắt đầu sau `5`. Chọn. `end = 7`.

Tổng `3`. Cell dưới làm đúng việc đó "bằng mắt" bằng code nhỏ nhất:

In [8]:
pairs = [[1, 100], [2, 3], [4, 5], [6, 7]]
by_end = sorted(pairs, key=lambda p: p[1])
print("sắp theo mút phải:", by_end)
end, chosen = float("-inf"), []
for a, b in by_end:
    if a > end:                      # bắt đầu sau mút phải hiện tại
        chosen.append([a, b])
        end = b
print("đã chọn:", chosen, "-> độ dài", len(chosen))

sắp theo mút phải: [[2, 3], [4, 5], [6, 7], [1, 100]]
đã chọn: [[2, 3], [4, 5], [6, 7]] -> độ dài 3


## 5. Cách 1 — Brute Force / baseline: thử mọi cách chọn (đệ quy chọn/bỏ)

**Ý tưởng.** Sắp các cặp theo mút trái. Với mỗi cặp ta có hai lựa chọn: **bỏ** hoặc **chọn** (chỉ được chọn nếu mút trái của nó **lớn hơn** mút phải của cặp đã chọn gần nhất). Hàm `go(i, last_end)` trả về độ dài chuỗi dài nhất xét các cặp từ `i` trở đi:

```
go(i, last_end) = max( go(i+1, last_end),                         # bỏ cặp i
                       1 + go(i+1, pairs[i].right)  nếu pairs[i].left > last_end )   # chọn cặp i
```

Cách này duyệt **mọi tập con** hợp lệ nên chắc chắn đúng, nhưng có tới `2^n` nhánh.

In [9]:
def chain_brute(pairs):
    ps = sorted(pairs)                                    # sắp theo mút trái (rồi mút phải)
    n = len(ps)

    def go(i, last_end):
        if i == n:
            return 0
        best = go(i + 1, last_end)                        # bỏ cặp i
        if ps[i][0] > last_end:                           # chọn cặp i nếu đi sau chuỗi hiện tại
            best = max(best, 1 + go(i + 1, ps[i][1]))
        return best

    return go(0, float("-inf"))


for pairs, expected in examples:
    got = chain_brute(pairs)
    print(pairs, "->", got)
    assert got == expected

[[1, 2], [2, 3], [3, 4]] -> 2
[[1, 2], [7, 8], [4, 5]] -> 3
[[1, 100], [2, 3], [4, 5]] -> 2
[[-10, -8], [8, 9], [-5, 0], [6, 10], [-6, -4], [1, 7], [9, 10], [-4, 7]] -> 4


**Độ phức tạp.** Thời gian **O(2^n)** (không dùng nhớ nên các nhánh lặp lại), bộ nhớ **O(n)** cho ngăn xếp đệ quy. Với `n = 1000` hoàn toàn không khả thi (chỉ dùng được tới `n` cỡ 20).

**Nút thắt.** Nhiều nhánh gọi lại **cùng một trạng thái** `(i, last_end)`. Và trạng thái "đã chọn cặp nào" có thể rút gọn: chỉ cần biết **mút phải của cặp cuối** (hoặc, cách khác, ta đang xét chuỗi **kết thúc tại cặp nào**). Đó là cơ sở cho quy hoạch động.

## 6. Ý tưởng thuật toán / bước đột phá

Có hai "bước đột phá" tương ứng hai họ lời giải:

**A. Quy hoạch động kiểu LIS.** Sắp theo mút trái. Gọi `dp[i]` là **độ dài chuỗi dài nhất kết thúc tại cặp `i`** (cặp `i` là cặp cuối). Cặp `i` có thể nối sau cặp `j < i` nếu `pairs[j].right < pairs[i].left`. Vậy:

```
dp[i] = 1 + max(dp[j] với j < i và pairs[j][1] < pairs[i][0]),   mặc định 1
đáp án = max(dp)
```

Cần sắp xếp trước để chắc rằng cặp "trước" trong chuỗi luôn có chỉ số nhỏ hơn.

**B. Tham lam (activity selection).** Sắp theo **mút phải** tăng dần. Duyệt từng cặp; nếu mút trái của cặp lớn hơn mút phải của cặp đã chọn cuối cùng thì chọn nó. Lý do (sẽ chứng minh ở mục 10): trong mọi lời giải tối ưu, có thể **thay cặp đầu tiên** bằng cặp **kết thúc sớm nhất** mà không mất độ dài, vì cặp mới kết thúc không muộn hơn nên không chặn thêm cặp nào; lặp lại cho phần còn lại.

Nối hai hướng: DP có thể tăng tốc bằng tìm kiếm nhị phân (mảng `tails`, cách 5), còn tham lam có hai dạng: sắp theo mút phải (cách 3) hoặc sắp theo mút trái rồi **co mút phải** (cách 4).

## 7. Các cách tiếp theo

Tổng cộng 5 cách: cách 1 (vét cạn, mục 5) và bốn cách dưới đây.

### 7.1. Cách 2 — Quy hoạch động O(n²) (LIS sau khi sắp theo mút trái)

**Giải thích từng đoạn.**

- Sắp: `pairs.sort()` (sắp theo mút trái, hòa thì theo mút phải).
- Mảng `dp = [1] * n`: mỗi cặp tự đứng một mình là chuỗi độ dài `1`.
- Với mỗi `i`, duyệt mọi `j < i`: nếu `pairs[j][1] < pairs[i][0]` thì `dp[i] = max(dp[i], dp[j] + 1)`.
- Đáp án: `max(dp)` (chuỗi có thể kết thúc ở cặp bất kỳ).

In [10]:
def chain_dp(pairs, verbose=False):
    ps = sorted(pairs)
    n = len(ps)
    dp = [1] * n
    for i in range(n):
        for j in range(i):
            if ps[j][1] < ps[i][0]:
                dp[i] = max(dp[i], dp[j] + 1)
    if verbose:
        for p, v in zip(ps, dp):
            print(f"  {p}: dp = {v}")
    return max(dp)


for pairs, expected in examples:
    got = chain_dp(pairs, verbose=(pairs == examples[0][0]))
    assert got == expected
print("OK")

  [1, 2]: dp = 1
  [2, 3]: dp = 1
  [3, 4]: dp = 2
OK


**Code nộp LeetCode (cách 2):**

In [11]:
class Solution2:
    def findLongestChain(self, pairs: List[List[int]]) -> int:
        pairs = sorted(pairs)
        n = len(pairs)
        dp = [1] * n
        for i in range(n):
            for j in range(i):
                if pairs[j][1] < pairs[i][0]:
                    dp[i] = max(dp[i], dp[j] + 1)
        return max(dp)


for pairs, expected in examples:
    assert Solution2().findLongestChain([p[:] for p in pairs]) == expected
print("Solution2 OK")

Solution2 OK


**Độ phức tạp cách 2.** Thời gian **O(n²)** (hai vòng lồng; với `n = 1000` là khoảng `5 × 10^5` phép so sánh, nhanh); bộ nhớ **O(n)** cho `dp` (cộng bản sao đã sắp).

### 7.2. Cách 3 — Tham lam: sắp theo **mút phải**, luôn chọn cặp kết thúc sớm nhất

**Giải thích từng đoạn.**

- Sắp theo mút phải: `pairs.sort(key=lambda p: p[1])`.
- `end = -inf` (mút phải của chuỗi hiện tại), `count = 0`.
- Duyệt từng cặp `[a, b]` theo thứ tự đó: nếu **`a > end`** thì cặp này đi sau chuỗi hiện tại, ta **chọn**: `count += 1; end = b`. Ngược lại **bỏ qua** (nó chồng lấn với cặp đã chọn).

Vì đã sắp theo mút phải, cặp ta gặp đầu tiên mà còn hợp lệ luôn là cặp **kết thúc sớm nhất** trong số cặp còn hợp lệ.

In [12]:
def chain_greedy_right(pairs, verbose=False):
    end, count = float("-inf"), 0
    for a, b in sorted(pairs, key=lambda p: p[1]):
        if a > end:
            count += 1
            end = b
            if verbose:
                print(f"  chọn [{a}, {b}] -> count = {count}, end = {end}")
        elif verbose:
            print(f"  bỏ   [{a}, {b}] vì {a} <= end = {end}")
    return count


for pairs, expected in examples:
    got = chain_greedy_right(pairs, verbose=(pairs == examples[1][0]))
    assert got == expected
print("OK")

  chọn [1, 2] -> count = 1, end = 2
  chọn [4, 5] -> count = 2, end = 5
  chọn [7, 8] -> count = 3, end = 8
OK


**Code nộp LeetCode (cách 3):**

In [13]:
class Solution3:
    def findLongestChain(self, pairs: List[List[int]]) -> int:
        end, count = float("-inf"), 0
        for a, b in sorted(pairs, key=lambda p: p[1]):
            if a > end:
                count += 1
                end = b
        return count


for pairs, expected in examples:
    assert Solution3().findLongestChain([p[:] for p in pairs]) == expected
print("Solution3 OK")

Solution3 OK


**Độ phức tạp cách 3.** Thời gian **O(n log n)** (sắp xếp), duyệt `O(n)`; bộ nhớ **O(n)** cho bản sao đã sắp (hoặc `O(1)` phụ nếu sắp tại chỗ `pairs.sort(...)`).

### 7.3. Cách 4 — Tham lam: sắp theo **mút trái**, nhưng "co" mút phải khi chồng lấn

**Ý tưởng.** Nếu sắp theo mút trái, ta không thể đơn giản "chọn nếu hợp lệ" (ví dụ 3: `[1,100]` đứng đầu sẽ chặn hết). Cách sửa: giữ `end` là mút phải của chuỗi hiện tại và **mỗi lần gặp cặp chồng lấn với chuỗi, ta coi như "đổi" cặp cuối của chuỗi sang cặp có mút phải nhỏ hơn**:

- Nếu `a > end`: cặp đi sau → `count += 1`, `end = b`.
- Ngược lại (chồng lấn): `end = min(end, b)` — giữ lại cặp kết thúc sớm hơn trong hai cặp chồng nhau; độ dài chuỗi **không đổi**.

Đây chính là cùng một chiến lược "kết thúc sớm nhất", chỉ viết cho thứ tự duyệt theo mút trái.

In [14]:
def chain_greedy_left(pairs, verbose=False):
    count, end = 0, float("-inf")
    for a, b in sorted(pairs):
        if a > end:
            count += 1
            end = b
            if verbose:
                print(f"  [{a}, {b}] đi sau -> count = {count}, end = {end}")
        else:
            end = min(end, b)                             # chồng lấn: giữ mút phải nhỏ hơn
            if verbose:
                print(f"  [{a}, {b}] chồng lấn -> co end = min(end, {b}) = {end}")
    return count


for pairs, expected in examples:
    got = chain_greedy_left(pairs, verbose=(pairs == examples[2][0]))
    assert got == expected
print("OK")

  [1, 100] đi sau -> count = 1, end = 100
  [2, 3] chồng lấn -> co end = min(end, 3) = 3
  [4, 5] đi sau -> count = 2, end = 5
OK


**Code nộp LeetCode (cách 4):**

In [15]:
class Solution4:
    def findLongestChain(self, pairs: List[List[int]]) -> int:
        count, end = 0, float("-inf")
        for a, b in sorted(pairs):
            if a > end:
                count += 1
                end = b
            else:
                end = min(end, b)
        return count


for pairs, expected in examples:
    assert Solution4().findLongestChain([p[:] for p in pairs]) == expected
print("Solution4 OK")

Solution4 OK


**Độ phức tạp cách 4.** Thời gian **O(n log n)**, bộ nhớ **O(n)** (bản sao đã sắp).

### 7.4. Cách 5 — Quy hoạch động + tìm kiếm nhị phân (mảng `tails`, như LIS)

**Ý tưởng.** Sắp theo mút trái. Duy trì list `tails` mà **`tails[L]` là mút phải nhỏ nhất của một chuỗi có độ dài `L + 1`** (xét các cặp đã duyệt). `tails` luôn **tăng chặt**.

Với cặp mới `[a, b]`: chuỗi dài nhất mà nó có thể nối sau là chuỗi có mút phải `< a`. Số chuỗi độ dài có mút phải `< a` là `idx = bisect_left(tails, a)` (các `tails[:idx]` đều `< a`). Vậy cặp này kéo dài chuỗi lên độ dài `idx + 1`, kết thúc tại `b`:

- nếu `idx == len(tails)`: chưa có chuỗi độ dài `idx + 1` → `tails.append(b)`;
- ngược lại: `tails[idx] = min(tails[idx], b)`.

Đáp án là `len(tails)`.

In [16]:
def chain_tails(pairs, verbose=False):
    tails = []
    for a, b in sorted(pairs):
        idx = bisect_left(tails, a)               # số mút phải trong tails nhỏ hơn a
        if idx == len(tails):
            tails.append(b)
        else:
            tails[idx] = min(tails[idx], b)
        if verbose:
            print(f"  [{a}, {b}] idx = {idx} -> tails = {tails}")
    return len(tails)


for pairs, expected in examples:
    got = chain_tails(pairs, verbose=(pairs == examples[3][0]))
    assert got == expected
print("OK")

  [-10, -8] idx = 0 -> tails = [-8]
  [-6, -4] idx = 1 -> tails = [-8, -4]
  [-5, 0] idx = 1 -> tails = [-8, -4]
  [-4, 7] idx = 1 -> tails = [-8, -4]
  [1, 7] idx = 2 -> tails = [-8, -4, 7]
  [6, 10] idx = 2 -> tails = [-8, -4, 7]
  [8, 9] idx = 3 -> tails = [-8, -4, 7, 9]
  [9, 10] idx = 3 -> tails = [-8, -4, 7, 9]
OK


**Code nộp LeetCode (cách 5):**

In [17]:
class Solution5:
    def findLongestChain(self, pairs: List[List[int]]) -> int:
        tails = []
        for a, b in sorted(pairs):
            idx = bisect_left(tails, a)
            if idx == len(tails):
                tails.append(b)
            else:
                tails[idx] = min(tails[idx], b)
        return len(tails)


for pairs, expected in examples:
    assert Solution5().findLongestChain([p[:] for p in pairs]) == expected
print("Solution5 OK")

Solution5 OK


**Độ phức tạp cách 5.** Thời gian **O(n log n)** (sắp xếp + `n` lần `bisect`); bộ nhớ **O(n)**. Lưu ý đây **không phải** "tham lam đơn thuần" mà là DP rút gọn: `tails` giữ thông tin cho **mọi** độ dài chuỗi, nên còn dùng được cho các biến thể có trọng số/đếm (ví dụ LIS).

## 8. TRACE từng bước

Ta trace trên ví dụ 4: `[[-10,-8],[8,9],[-5,0],[6,10],[-6,-4],[1,7],[9,10],[-4,7]]` (đáp án `4`).

### 8.1. Trace cách 3 (tham lam theo mút phải)

Bảng trạng thái:

| Cột | Ý nghĩa |
|---|---|
| `cặp [a, b]` | cặp đang xét, theo thứ tự mút phải tăng dần |
| `a > end?` | có đi sau chuỗi hiện tại không |
| `quyết định` | chọn hay bỏ |
| `end`, `count` | mút phải của chuỗi hiện tại và độ dài hiện tại |

In [18]:
def trace_greedy_right(pairs):
    ordered = sorted(pairs, key=lambda p: p[1])
    print("sắp theo mút phải:", ordered)
    end, count, chain = float("-inf"), 0, []
    for a, b in ordered:
        ok = a > end
        if ok:
            count += 1
            end = b
            chain.append([a, b])
        print(f"cặp [{a:>3}, {b:>3}] | a > end? {str(ok):<5} ({a} > {end}) | {'CHỌN' if ok else 'bỏ  '} | end = {end}, count = {count}")
    print("chuỗi tìm được:", chain)
    return count

print("kết quả:", trace_greedy_right(examples[3][0]))

sắp theo mút phải: [[-10, -8], [-6, -4], [-5, 0], [1, 7], [-4, 7], [8, 9], [6, 10], [9, 10]]
cặp [-10,  -8] | a > end? True  (-10 > -8) | CHỌN | end = -8, count = 1
cặp [ -6,  -4] | a > end? True  (-6 > -4) | CHỌN | end = -4, count = 2
cặp [ -5,   0] | a > end? False (-5 > -4) | bỏ   | end = -4, count = 2
cặp [  1,   7] | a > end? True  (1 > 7) | CHỌN | end = 7, count = 3
cặp [ -4,   7] | a > end? False (-4 > 7) | bỏ   | end = 7, count = 3
cặp [  8,   9] | a > end? True  (8 > 9) | CHỌN | end = 9, count = 4
cặp [  6,  10] | a > end? False (6 > 9) | bỏ   | end = 9, count = 4
cặp [  9,  10] | a > end? False (9 > 9) | bỏ   | end = 9, count = 4
chuỗi tìm được: [[-10, -8], [-6, -4], [1, 7], [8, 9]]
kết quả: 4


Quan sát: `[-10,-8]` kết thúc sớm nhất nên được chọn trước (`end = -8`). `[-6,-4]` có `-6 > -8` nên được chọn (`end = -4`). `[-5,0]` có `-5 > -4` sai nên bị bỏ. `[1,7]` được chọn (`end = 7`), `[-4,7]` bị bỏ, `[8,9]` được chọn (`end = 9`). Hai cặp cuối `[6,10]` và `[9,10]` bị bỏ; đặc biệt `[9,10]` bị loại vì `9 > 9` sai (chạm nhau). Chuỗi tìm được: `[-10,-8] -> [-6,-4] -> [1,7] -> [8,9]`, độ dài `4`.

### 8.2. Trace cách 2 (bảng `dp` theo thứ tự mút trái)

In [19]:
def trace_dp(pairs):
    ps = sorted(pairs)
    n = len(ps)
    dp = [1] * n
    parent = [-1] * n
    print("sắp theo mút trái:", ps)
    for i in range(n):
        for j in range(i):
            if ps[j][1] < ps[i][0] and dp[j] + 1 > dp[i]:
                dp[i] = dp[j] + 1
                parent[i] = j
        why = f"nối sau {ps[parent[i]]}" if parent[i] >= 0 else "đứng một mình"
        print(f"i = {i}: {ps[i]!s:<10} dp = {dp[i]}  ({why})")
    best = max(range(n), key=lambda i: dp[i])
    chain = []
    while best != -1:
        chain.append(ps[best])
        best = parent[best]
    print("max(dp) =", max(dp), "| chuỗi (đảo ngược lại):", chain[::-1])
    return max(dp)

trace_dp(examples[3][0])

sắp theo mút trái: [[-10, -8], [-6, -4], [-5, 0], [-4, 7], [1, 7], [6, 10], [8, 9], [9, 10]]
i = 0: [-10, -8]  dp = 1  (đứng một mình)
i = 1: [-6, -4]   dp = 2  (nối sau [-10, -8])
i = 2: [-5, 0]    dp = 2  (nối sau [-10, -8])
i = 3: [-4, 7]    dp = 2  (nối sau [-10, -8])
i = 4: [1, 7]     dp = 3  (nối sau [-6, -4])
i = 5: [6, 10]    dp = 3  (nối sau [-6, -4])
i = 6: [8, 9]     dp = 4  (nối sau [1, 7])
i = 7: [9, 10]    dp = 4  (nối sau [1, 7])
max(dp) = 4 | chuỗi (đảo ngược lại): [[-10, -8], [-6, -4], [1, 7], [8, 9]]


4

Đọc bảng `dp`: `dp[i]` là độ dài chuỗi dài nhất **kết thúc tại** cặp `i`; cột `parent` cho phép lần ngược để dựng chuỗi cụ thể.

### 8.3. Trace cách 4 (tham lam theo mút trái có co `end`)

In [20]:
def trace_greedy_left(pairs):
    ps = sorted(pairs)
    print("sắp theo mút trái:", ps)
    count, end = 0, float("-inf")
    for a, b in ps:
        if a > end:
            count += 1
            end = b
            act = f"đi sau -> count = {count}, end = {end}"
        else:
            old = end
            end = min(end, b)
            act = f"chồng lấn -> end = min({old}, {b}) = {end}"
        print(f"[{a:>3}, {b:>3}]: {act}")
    return count

print("kết quả:", trace_greedy_left(examples[3][0]))

sắp theo mút trái: [[-10, -8], [-6, -4], [-5, 0], [-4, 7], [1, 7], [6, 10], [8, 9], [9, 10]]
[-10,  -8]: đi sau -> count = 1, end = -8
[ -6,  -4]: đi sau -> count = 2, end = -4
[ -5,   0]: chồng lấn -> end = min(-4, 0) = -4
[ -4,   7]: chồng lấn -> end = min(-4, 7) = -4
[  1,   7]: đi sau -> count = 3, end = 7
[  6,  10]: chồng lấn -> end = min(7, 10) = 7
[  8,   9]: đi sau -> count = 4, end = 9
[  9,  10]: chồng lấn -> end = min(9, 10) = 9
kết quả: 4


### 8.4. Trace cách 5 (`tails`)

Phần in `verbose=True` ở mục 7.4 đã cho thấy `tails` thay đổi thế nào. Ở đây ta trace lại trên một ví dụ nhỏ có phần thay thế rõ ràng, `[[1,10],[2,3],[4,5],[6,7]]`:

In [21]:
print("kết quả:", chain_tails([[1, 10], [2, 3], [4, 5], [6, 7]], verbose=True))

  [1, 10] idx = 0 -> tails = [10]
  [2, 3] idx = 0 -> tails = [3]
  [4, 5] idx = 1 -> tails = [3, 5]
  [6, 7] idx = 2 -> tails = [3, 5, 7]
kết quả: 3


Đọc trace: `[1,10]` mở chuỗi độ dài 1 với mút phải `10`. Cặp `[2,3]`: `bisect_left([10], 2) = 0` nên **thay** `tails[0] = min(10, 3) = 3` (chuỗi độ dài 1 kết thúc sớm hơn là tốt hơn). Cặp `[4,5]` nối sau chuỗi độ dài 1 (mút phải `3 < 4`) → `tails = [3, 5]`. Cặp `[6,7]` → `[3, 5, 7]`. Đáp án 3.

## 9. Code nộp LeetCode

Chọn **cách 3 (tham lam theo mút phải)**: ngắn nhất, nhanh nhất, và có chứng minh gọn. Đây là code đúng class/method/signature (trên LeetCode `List` có sẵn; ở đây ta đã import ở trên):

In [22]:
class Solution:
    def findLongestChain(self, pairs: List[List[int]]) -> int:
        end = float("-inf")                       # mút phải của chuỗi hiện tại (chưa chọn gì)
        count = 0
        for a, b in sorted(pairs, key=lambda p: p[1]):   # kết thúc sớm trước
            if a > end:                            # đi sau chuỗi hiện tại (nhỏ hơn HẲN)
                count += 1
                end = b
        return count


for pairs, expected in examples:
    got = Solution().findLongestChain(pairs)
    print(pairs, "->", got)
    assert got == expected

[[1, 2], [2, 3], [3, 4]] -> 2
[[1, 2], [7, 8], [4, 5]] -> 3
[[1, 100], [2, 3], [4, 5]] -> 2
[[-10, -8], [8, 9], [-5, 0], [6, 10], [-6, -4], [1, 7], [9, 10], [-4, 7]] -> 4


## 10. Vì sao thuật toán đúng

**Tham lam theo mút phải (cách 3).** Sắp các cặp theo mút phải tăng dần `p1, p2, ..., pn` (`right` không giảm). Khẳng định: **tồn tại một chuỗi tối ưu chứa `p1`** (cặp kết thúc sớm nhất).

*Chứng minh (hoán đổi — exchange argument).* Lấy một chuỗi tối ưu `q1 -> q2 -> ... -> qm` (xếp tăng dần). Nếu `q1 = p1` xong. Ngược lại thay `q1` bằng `p1`: ta có `right(p1) ≤ right(q1)` nên `right(p1) ≤ right(q1) < left(q2)`, tức `p1 -> q2` vẫn hợp lệ. Độ dài không đổi, và chuỗi mới chứa `p1`. ∎

Sau khi chọn `p1`, các cặp còn dùng được là những cặp có `left > right(p1)`; bài toán con trên các cặp đó **có cùng cấu trúc** (lặp lại lập luận). Duyệt theo thứ tự mút phải và "chọn nếu `left > end`" chính là thực hiện lập luận đó: cặp đầu tiên gặp có `left > end` là cặp kết thúc sớm nhất trong các cặp còn hợp lệ.

**Vì sao `a > end` đúng điều kiện.** Đề đòi `right(trước) < left(sau)`, nên cặp mới hợp lệ khi `left > end` với `end` là mút phải của cặp cuối chuỗi.

**DP (cách 2).** `dp[i]` là độ dài chuỗi dài nhất kết thúc tại cặp `i` (theo thứ tự mút trái). Một chuỗi kết thúc tại `i` có cặp áp chót `j` thỏa `right(j) < left(i)`, và nó phải có `left(j) < right(j) < left(i)`, tức `j` đứng **trước** `i` trong thứ tự mút trái ⇒ `j < i`. Phần chuỗi còn lại là chuỗi kết thúc tại `j`, nên `dp[i] = 1 + max dp[j]`; đáp án là `max dp`.

**Cách 4 (co mút phải).** Sắp theo mút trái; giả sử chuỗi hiện tại kết thúc bằng một cặp có mút phải `end`, và cặp mới `[a, b]` chồng lấn với nó (`a ≤ end`). Vì sắp theo mút trái nên `a` không nhỏ hơn mút trái của cặp cuối chuỗi, mà mút trái đó lớn hơn mút phải của cặp áp chót; do đó `[a, b]` cũng đi sau cặp áp chót. Vậy ta có thể **thay** cặp cuối bằng `[a, b]` mà độ dài không đổi. Nếu `b < end` thì mút phải của chuỗi giảm xuống `b`, nếu không thì giữ nguyên: tổng quát là `end = min(end, b)`. Mút phải nhỏ hơn không bao giờ làm các cặp sau khó nối hơn, nên chiến lược này không kém hơn.

**Cách 5 (`tails`).** Bất biến: `tails[L]` là mút phải nhỏ nhất của một chuỗi độ dài `L + 1` trên các cặp đã duyệt; `tails` tăng chặt (một chuỗi độ dài `L + 2` kết thúc tại `e` chứa một chuỗi độ dài `L + 1` kết thúc `< e`). Với cặp `[a, b]`, độ dài tốt nhất của chuỗi kết thúc tại nó là `idx + 1` với `idx` = số phần tử `tails` nhỏ hơn `a`. Cập nhật `tails[idx] = min(tails[idx], b)` giữ đúng bất biến.

**Cách 1** đúng theo định nghĩa (duyệt mọi tập con hợp lệ).

## 11. Phân tích Time / Space Complexity

Gọi `n` là số cặp (`n ≤ 1000`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn chọn/bỏ | O(2^n) | O(n) | chỉ dùng được `n` nhỏ (làm oracle) |
| 2. DP kiểu LIS | O(n²) | O(n) | tổng quát, mở rộng được cho bài có trọng số |
| 3. Tham lam theo mút phải | O(n log n) | O(n) hoặc O(1) phụ | **bản nộp**, ngắn nhất |
| 4. Tham lam theo mút trái + co `end` | O(n log n) | O(n) | cùng chiến lược, thứ tự duyệt khác |
| 5. DP + `bisect` (`tails`) | O(n log n) | O(n) | gốc LIS, mở rộng được |

**Chi tiết cách 3.** `sorted` chiếm `O(n log n)`; vòng duyệt `O(n)` với phép toán `O(1)`. Với `n = 1000` mỗi cách chạy trong vài ms (cách 2 vài chục ms). Giới hạn `n ≤ 1000` cho phép cả `O(n²)`; mức `O(n log n)` là mức "đẹp" mà đề muốn bạn nhận ra.

**Chi tiết cách 5.** `bisect_left` là `O(log len(tails))`, nhưng `tails[idx] = ...` và `append` là `O(1)`; không có phép chèn vào giữa list, nên mỗi cặp tốn `O(log n)`.

## 12. Bộ test

Ta kiểm tra theo nhiều lớp:

1. Ví dụ của đề (và các ví dụ tự đặt) cho cả 5 cách + bản nộp.
2. Edge case: `n = 1`, cặp trùng nhau, cặp lồng nhau, chạm nhau ở một điểm, toàn số âm, biên `-1000` và `1000`.
3. **Phản ví dụ** của các ý tưởng sai.
4. Đối chiếu với **oracle độc lập** (duyệt mọi tập con bằng mặt nạ bit và kiểm tra chuỗi hợp lệ) trên input ngẫu nhiên có seed, đúng ràng buộc.
5. Test lớn `n = 1000` và so sánh thời gian.

In [23]:
def oracle_subsets(pairs):
    """Oracle độc lập (n <= ~14): thử mọi tập con, xếp theo mút phải, kiểm tra từng cặp liên tiếp."""
    n = len(pairs)
    best = 0
    for mask in range(1, 1 << n):
        chosen = [pairs[i] for i in range(n) if mask >> i & 1]
        if len(chosen) <= best:
            continue
        chosen.sort(key=lambda p: (p[1], p[0]))
        if all(chosen[i][1] < chosen[i + 1][0] for i in range(len(chosen) - 1)):
            best = len(chosen)
    return best


for pairs, expected in examples:
    assert oracle_subsets(pairs) == expected
print("oracle khớp ví dụ")

solvers = {
    "brute": chain_brute,
    "dp": chain_dp,
    "greedy_right": chain_greedy_right,
    "greedy_left": chain_greedy_left,
    "tails": chain_tails,
    "Solution2": lambda p: Solution2().findLongestChain(p),
    "Solution3": lambda p: Solution3().findLongestChain(p),
    "Solution4": lambda p: Solution4().findLongestChain(p),
    "Solution5": lambda p: Solution5().findLongestChain(p),
    "Solution": lambda p: Solution().findLongestChain(p),
}
for pairs, expected in examples:
    for name, fn in solvers.items():
        assert fn([p[:] for p in pairs]) == expected, (name, pairs)
print("tất cả cách khớp ví dụ:", list(solvers))

oracle khớp ví dụ
tất cả cách khớp ví dụ: ['brute', 'dp', 'greedy_right', 'greedy_left', 'tails', 'Solution2', 'Solution3', 'Solution4', 'Solution5', 'Solution']


In [24]:
# --- Edge case ---
edge_cases = [
    [[0, 1]],                                              # n = 1
    [[1, 2], [1, 2]],                                      # trùng nhau -> 1
    [[1, 2], [1, 2], [3, 4]],                              # trùng + một cặp sau -> 2
    [[1, 10], [2, 9], [3, 8]],                             # lồng nhau -> 1
    [[1, 2], [2, 3], [3, 4], [4, 5]],                      # chạm nhau -> 2 ([1,2],[3,4]) hoặc ([2,3],[4,5])
    [[-1000, -999], [-5, -4], [999, 1000]],                # biên + số âm -> 3
    [[-1000, 1000]],                                       # khoảng rộng nhất
    [[-3, -2], [-2, -1], [-1, 0], [0, 1]],                 # toàn chạm nhau
    [[5, 6], [1, 2], [3, 4]],                              # input không sắp
    [[1, 100], [2, 3], [4, 5], [6, 7]],                    # cặp lớn che mọi cặp -> 3
    [[1, 3], [2, 4], [3, 5], [4, 6], [5, 7]],              # các cặp trải đều
]
for pairs in edge_cases:
    want = oracle_subsets(pairs)
    for name, fn in solvers.items():
        got = fn([p[:] for p in pairs])
        assert got == want, (name, pairs, got, want)
    print(pairs, "->", want)

[[0, 1]] -> 1
[[1, 2], [1, 2]] -> 1
[[1, 2], [1, 2], [3, 4]] -> 2
[[1, 10], [2, 9], [3, 8]] -> 1
[[1, 2], [2, 3], [3, 4], [4, 5]] -> 2
[[-1000, -999], [-5, -4], [999, 1000]] -> 3
[[-1000, 1000]] -> 1
[[-3, -2], [-2, -1], [-1, 0], [0, 1]] -> 2
[[5, 6], [1, 2], [3, 4]] -> 3
[[1, 100], [2, 3], [4, 5], [6, 7]] -> 3
[[1, 3], [2, 4], [3, 5], [4, 6], [5, 7]] -> 2


**Phản ví dụ.** Các ý tưởng "nghe có vẻ đúng" nhưng sai:

- *Sai A:* dùng `a >= end` (cho phép chạm nhau).
- *Sai B:* sắp theo **mút trái** rồi "chọn nếu hợp lệ" (không co `end`).
- *Sai C:* sắp theo mút phải nhưng khởi tạo `end = 0`.
- *Sai D:* sắp theo **độ dài** (cặp ngắn trước) rồi chọn nếu không chồng lấn với **bất kỳ cặp đã chọn**.

In [25]:
def wrong_touching(pairs):                      # sai A
    end, count = float("-inf"), 0
    for a, b in sorted(pairs, key=lambda p: p[1]):
        if a >= end:
            count += 1
            end = b
    return count

def wrong_left_no_shrink(pairs):                # sai B
    end, count = float("-inf"), 0
    for a, b in sorted(pairs):
        if a > end:
            count += 1
            end = b
    return count

def wrong_init_zero(pairs):                     # sai C
    end, count = 0, 0
    for a, b in sorted(pairs, key=lambda p: p[1]):
        if a > end:
            count += 1
            end = b
    return count

def wrong_by_length(pairs):                     # sai D
    chosen = []
    for a, b in sorted(pairs, key=lambda p: p[1] - p[0]):
        if all(b < c or d < a for c, d in chosen):      # không chồng lấn với mọi cặp đã chọn
            chosen.append([a, b])
    return len(chosen)


tests = {
    "A": (wrong_touching, [[1, 2], [2, 3], [3, 4]]),            # đúng 2, sai ra 3
    "B": (wrong_left_no_shrink, [[1, 100], [2, 3], [4, 5]]),    # đúng 2, sai ra 1
    "C": (wrong_init_zero, [[-5, -3], [-2, -1]]),               # đúng 2, sai ra 0
}
for label, (fn, pairs) in tests.items():
    print(f"sai {label}: {fn(pairs)} | đúng: {Solution().findLongestChain(pairs)} | pairs = {pairs}")
    assert fn(pairs) != Solution().findLongestChain(pairs)

# Sai D: tìm một phản ví dụ nhỏ bằng tìm kiếm có seed (đối chiếu với oracle độc lập)
rng = random.Random(646)
found = None
for _ in range(20000):
    n = rng.randint(3, 6)
    ps = []
    for _ in range(n):
        a = rng.randint(0, 9)
        ps.append([a, rng.randint(a + 1, 10)])
    if wrong_by_length(ps) != oracle_subsets(ps):
        found = ps
        break
print("sai D:", found, "-> sai ra", wrong_by_length(found), "| đúng", oracle_subsets(found))
assert found is not None

sai A: 3 | đúng: 2 | pairs = [[1, 2], [2, 3], [3, 4]]
sai B: 1 | đúng: 2 | pairs = [[1, 100], [2, 3], [4, 5]]
sai C: 0 | đúng: 2 | pairs = [[-5, -3], [-2, -1]]
sai D: [[4, 7], [0, 7], [0, 2], [7, 9], [8, 10], [8, 10]] -> sai ra 2 | đúng 3


Giải thích: **A** cho `3` vì coi `[1,2] -> [2,3] -> [3,4]` là chuỗi, trong khi `2 < 2` là sai. **B** chọn `[1,100]` rồi bị chặn mọi cặp sau. **C** khởi tạo `end = 0` làm mọi cặp có mút trái `≤ 0` bị loại oan. **D** (chọn cặp ngắn trước) sai vì **độ ngắn không phải tiêu chí đúng**: một cặp ngắn nhưng nằm giữa có thể chặn hai cặp hai bên.

**Đối chiếu nhiều input ngẫu nhiên (có seed, đúng ràng buộc).** Ta sinh `n` từ `1` đến `12` cặp với miền giá trị nhỏ (để hay trùng/chạm nhau) và miền rộng `[-1000, 1000]`.

In [26]:
def random_pairs(rng, n, lo, hi):
    out = []
    for _ in range(n):
        a = rng.randint(lo, hi - 1)
        b = rng.randint(a + 1, hi)                 # đảm bảo left < right
        out.append([a, b])
    return out


rng = random.Random(6460)
count = 0
for _ in range(500):
    n = rng.randint(1, 12)
    lo, hi = rng.choice([(0, 6), (0, 12), (-5, 5), (-1000, 1000)])
    pairs = random_pairs(rng, n, lo, hi)
    want = oracle_subsets(pairs)
    for name, fn in solvers.items():
        original = [p[:] for p in pairs]
        got = fn(original)
        assert got == want, (name, pairs, got, want)
    count += 1
print("đối chiếu", count, "input ngẫu nhiên (n <= 12): tất cả cách khớp oracle tập con")

# n lớn: bỏ oracle vét cạn, đối chiếu DP O(n^2) với các cách còn lại
for _ in range(60):
    n = rng.randint(100, 1000)
    lo, hi = rng.choice([(-1000, 1000), (-50, 50), (0, 300)])
    pairs = random_pairs(rng, n, lo, hi)
    ref = chain_dp(pairs)
    for name in ("greedy_right", "greedy_left", "tails", "Solution2", "Solution3", "Solution4", "Solution5", "Solution"):
        assert solvers[name]([p[:] for p in pairs]) == ref, (name, n)
print("60 input n từ 100 đến 1000: các cách nhanh khớp DP O(n^2)")

# input không bị sửa đổi (bản nộp dùng sorted, không sort tại chỗ)
pairs = random_pairs(rng, 50, -20, 20)
snapshot = [p[:] for p in pairs]
Solution().findLongestChain(pairs)
assert pairs == snapshot
print("bản nộp không làm thay đổi input")

đối chiếu 500 input ngẫu nhiên (n <= 12): tất cả cách khớp oracle tập con


60 input n từ 100 đến 1000: các cách nhanh khớp DP O(n^2)
bản nộp không làm thay đổi input


In [27]:
# Test lớn nhất theo ràng buộc: n = 1000 và so sánh thời gian
rng = random.Random(1000)
pairs = random_pairs(rng, 1000, -1000, 1000)
for name in ("dp", "greedy_right", "greedy_left", "tails"):
    t0 = time.perf_counter()
    r = solvers[name]([p[:] for p in pairs])
    print(f"{name:>13}: {(time.perf_counter() - t0) * 1000:8.2f} ms -> {r}")

# chuỗi cực đại: 1000 cặp rời nhau -> 1000 ... không thể (miền chỉ 2001 điểm): dùng [2i, 2i+1] cho 1000 cặp
chain_all = [[2 * i - 1000, 2 * i - 999] for i in range(1000)]
assert max(p[1] for p in chain_all) <= 1000
for name in ("dp", "greedy_right", "greedy_left", "tails", "Solution"):
    assert solvers[name]([p[:] for p in chain_all]) == 1000
print("1000 cặp rời nhau từng đôi -> 1000: OK")

           dp:    47.05 ms -> 43
 greedy_right:     0.29 ms -> 43
  greedy_left:     0.42 ms -> 43
        tails:     0.48 ms -> 43
1000 cặp rời nhau từng đôi -> 1000: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `>=` thay vì `>` (cho phép chạm nhau)

Đề nói "`b < c`" tức **nhỏ hơn hẳn**; hai cặp chạm ở một điểm (`[1,2]` và `[2,3]`) **không** nối được.

In [28]:
pairs = [[1, 2], [2, 3], [3, 4]]
print("sai  (>=):", wrong_touching(pairs))
print("đúng (>) :", Solution().findLongestChain(pairs))

sai  (>=): 3
đúng (>) : 2


**Vì sao sai.** Với `>=`, ta chấp nhận `[1,2] -> [2,3] -> [3,4]` (độ dài 3) trong khi thực tế `2 < 2` sai nên chỉ được `2`. **Cách sửa:** đúng dấu `a > end` (hoặc `end < a`).

### Lỗi 2: sắp theo mút trái rồi tham lam "chọn nếu hợp lệ"

Khi sắp theo mút trái, cặp "to" đứng đầu có thể **chặn** nhiều cặp nhỏ phía sau.

In [29]:
pairs = [[1, 100], [2, 3], [4, 5]]
print("sai  :", wrong_left_no_shrink(pairs), "(chọn [1,100] rồi hết cặp nào hợp lệ)")
print("đúng :", Solution().findLongestChain(pairs))
print("cách 4 (có co end):", chain_greedy_left(pairs))

sai  : 1 (chọn [1,100] rồi hết cặp nào hợp lệ)
đúng : 2
cách 4 (có co end): 2


**Vì sao sai.** Tham lam muốn **kết thúc sớm nhất**, không phải **bắt đầu sớm nhất**. **Cách sửa:** sắp theo **mút phải** (cách 3), hoặc nếu sắp theo mút trái thì phải **co `end = min(end, b)`** khi chồng lấn (cách 4).

### Lỗi 3: khởi tạo "mút phải hiện tại" bằng `0`

Mút trái có thể âm, nên `end = 0` làm các cặp nằm hoàn toàn ở phía âm bị loại oan.

In [30]:
pairs = [[-5, -3], [-2, -1]]
print("sai  (end = 0):", wrong_init_zero(pairs), "(đúng phải là 2)")
print("đúng (-inf)   :", Solution().findLongestChain(pairs))

sai  (end = 0): 0 (đúng phải là 2)
đúng (-inf)   : 2


**Vì sao sai.** `a > 0` sai với mọi `a` âm, nên không cặp nào được chọn. **Cách sửa:** `end = float("-inf")` (hoặc `-10**9`, hoặc dùng cờ "chưa chọn").

### Lỗi 4: DP lấy `dp[-1]` thay vì `max(dp)`

`dp[i]` là chuỗi dài nhất **kết thúc tại** cặp `i`, không phải "xét tới cặp `i`". Cặp cuối trong thứ tự sắp **không nhất thiết** là cặp cuối của chuỗi dài nhất.

In [31]:
def wrong_dp_last(pairs):
    ps = sorted(pairs, key=lambda p: p[1])            # sắp theo mút phải
    n = len(ps)
    dp = [1] * n
    for i in range(n):
        for j in range(i):
            if ps[j][1] < ps[i][0]:
                dp[i] = max(dp[i], dp[j] + 1)
    return dp[-1]                                     # SAI: phải là max(dp)

pairs = [[1, 2], [3, 4], [0, 5]]
print("sai  (dp[-1]):", wrong_dp_last(pairs))
print("đúng (max)   :", Solution2().findLongestChain(pairs))

sai  (dp[-1]): 1
đúng (max)   : 2


**Vì sao sai.** Theo mút phải: `[1,2], [3,4], [0,5]` có `dp = [1, 2, 1]`: cặp cuối `[0,5]` chỉ có chuỗi độ dài `1`, trong khi chuỗi dài nhất (độ dài `2`) kết thúc ở `[3,4]`. **Cách sửa:** `return max(dp)`.

### Lỗi 5: quy hoạch động mà **quên sắp xếp**

DP kiểu LIS chỉ đúng nếu "cặp trước trong chuỗi" luôn có **chỉ số nhỏ hơn**. Nếu input không sắp, ta có thể bỏ lỡ các cặp nối.

In [32]:
def wrong_dp_unsorted(pairs):
    n = len(pairs)
    dp = [1] * n
    for i in range(n):
        for j in range(i):
            if pairs[j][1] < pairs[i][0]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)

pairs = [[3, 4], [1, 2]]
print("sai  (không sắp):", wrong_dp_unsorted(pairs))
print("đúng            :", Solution2().findLongestChain(pairs))

sai  (không sắp): 1
đúng            : 2


**Vì sao sai.** `[1,2]` nằm **sau** `[3,4]` trong mảng nên khi tính `dp[1]` ta so `[3,4]` với `[1,2]` (`4 < 1` sai) và bỏ lỡ cặp hợp lệ `[1,2] -> [3,4]`. **Cách sửa:** `pairs.sort()` trước khi DP (đề cho phép chọn thứ tự bất kỳ).

### Lỗi 6: nhầm bài với "sắp theo độ dài cặp"

Cặp **ngắn** không có nghĩa là **tốt**: một cặp ngắn nằm giữa có thể chặn hai cặp hai bên (xem `wrong_by_length` ở mục 12).

In [33]:
print("phản ví dụ tìm được ở mục 12:", found)
print("sai (theo độ dài):", wrong_by_length(found), "| đúng:", Solution().findLongestChain(found))

phản ví dụ tìm được ở mục 12: [[4, 7], [0, 7], [0, 2], [7, 9], [8, 10], [8, 10]]
sai (theo độ dài): 2 | đúng: 3


**Cách sửa.** Tiêu chí đúng là **mút phải nhỏ nhất**, không phải độ dài. Mỗi bài tham lam cần một **chứng minh hoán đổi**; nếu bạn không chứng minh được thì hãy nghi ngờ tiêu chí đó và kiểm tra bằng oracle vét cạn trên input nhỏ.

## 14. Mẹo & nhận dạng pattern

- **"Chọn nhiều khoảng/công việc không chồng nhau nhất"** → **sắp theo mút phải**, tham lam chọn cái kết thúc sớm nhất. Đây là activity selection; các bài họ hàng: *Non-overlapping Intervals (435)*, *Minimum Number of Arrows to Burst Balloons (452)*, *Meeting Rooms*.
- **"Dãy con tăng dài nhất" trên đối tượng 2 chiều** → sắp theo một chiều rồi LIS trên chiều còn lại (*Russian Doll Envelopes (354)*, bài này).
- **Phân biệt `<` và `<=`** ngay từ đề: "chạm nhau" có được tính là chồng không? Test bằng cặp `[1,2],[2,3]`.
- **Khởi tạo "chưa có gì" bằng `-inf`** khi giá trị có thể âm.
- **Tham lam hay DP?** Ràng buộc `n ≤ 1000` gợi ý `O(n²)` DP cũng qua; nhưng cấu trúc "đoạn không giao nhau" gợi ý tham lam `O(n log n)`. Hãy viết DP/vét cạn làm oracle để kiểm tham lam.
- **Cách chứng minh tham lam**: "hoán đổi": lấy một nghiệm tối ưu bất kỳ rồi chứng minh có thể thay phần tử đầu bằng lựa chọn tham lam mà không xấu đi.
- **Trong Python**: `sorted(pairs, key=lambda p: p[1])` và `float("-inf")` là cặp đôi thân quen của các bài khoảng.

## 15. Biến thể

### 15.1. Dựng **chính chuỗi** dài nhất (không chỉ độ dài)

Với tham lam theo mút phải, chuỗi chính là các cặp được chọn.

In [34]:
def longest_chain_path(pairs):
    chain, end = [], float("-inf")
    for a, b in sorted(pairs, key=lambda p: p[1]):
        if a > end:
            chain.append([a, b])
            end = b
    return chain


path = longest_chain_path([[1, 2], [7, 8], [4, 5]])
print(path)                                                 # [[1, 2], [4, 5], [7, 8]]
assert path == [[1, 2], [4, 5], [7, 8]]
assert all(path[i][1] < path[i + 1][0] for i in range(len(path) - 1))

[[1, 2], [4, 5], [7, 8]]


### 15.2. Cho phép **chạm nhau** (`b <= c`)

Chỉ đổi dấu `>` thành `>=`; mọi lập luận hoán đổi vẫn đúng.

In [35]:
def longest_chain_touching(pairs):
    end, count = float("-inf"), 0
    for a, b in sorted(pairs, key=lambda p: p[1]):
        if a >= end:
            count += 1
            end = b
    return count

print(longest_chain_touching([[1, 2], [2, 3], [3, 4]]))     # 3
assert longest_chain_touching([[1, 2], [2, 3], [3, 4]]) == 3
assert longest_chain_touching([[1, 3], [2, 4]]) == 1

3


### 15.3. Ít cặp cần **xóa** nhất để các cặp còn lại không chồng nhau (LeetCode 435)

Đáp án = `n - (số cặp không chồng tối đa)`; với 435 chạm nhau được tính là **không** chồng, nên dùng bản `>=`.

In [36]:
def erase_overlap_intervals(intervals):
    return len(intervals) - longest_chain_touching(intervals)

print(erase_overlap_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]))     # 1
assert erase_overlap_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]) == 1
assert erase_overlap_intervals([[1, 2], [1, 2], [1, 2]]) == 2
assert erase_overlap_intervals([[1, 2], [2, 3]]) == 0

1


### 15.4. LIS thường (LeetCode 300) bằng cùng mảng `tails`

Cách 5 chính là LIS áp dụng cho "mút phải < mút trái". Với dãy số, `tails` dùng `bisect_left` (tăng chặt).

In [37]:
def lis_length(nums):
    tails = []
    for x in nums:
        i = bisect_left(tails, x)
        if i == len(tails):
            tails.append(x)
        else:
            tails[i] = x
    return len(tails)

print(lis_length([10, 9, 2, 5, 3, 7, 101, 18]))             # 4
assert lis_length([10, 9, 2, 5, 3, 7, 101, 18]) == 4
assert lis_length([0, 1, 0, 3, 2, 3]) == 4
assert lis_length([7, 7, 7]) == 1

4


### 15.5. Biến thể để tự nghĩ

- **Có trọng số** (mỗi cặp có giá trị, tối đa tổng giá trị — *Maximum Profit in Job Scheduling (1235)*): tham lam không còn đúng, dùng DP + `bisect`.
- **Đếm số chuỗi dài nhất**: DP kiểu LIS có thêm mảng đếm (*Number of Longest Increasing Subsequence (673)*).
- **Mút trái/phải cùng lớn hơn** (hai chiều đều tăng): *Russian Doll Envelopes (354)* cần sắp theo `(w tăng, h giảm)` rồi LIS trên `h`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn chọn/bỏ | thử mọi tập con hợp lệ | O(2^n) | O(n) | chỉ làm oracle, `n ≤ ~20` |
| 2. DP kiểu LIS | `dp[i] = 1 + max dp[j]` nếu `right_j < left_i` | O(n²) | O(n) | khi cần mở rộng (trọng số, đếm); `n ≤ 1000` |
| 3. Tham lam theo mút phải | luôn chọn cặp kết thúc sớm nhất còn hợp lệ | O(n log n) | O(n) | **cách nộp mặc định** |
| 4. Tham lam theo mút trái, co `end` | chồng lấn thì `end = min(end, b)` | O(n log n) | O(n) | khi dữ liệu đã/chỉ sắp theo mút trái |
| 5. DP + `bisect` (`tails`) | mút phải nhỏ nhất cho mỗi độ dài chuỗi | O(n log n) | O(n) | dạng LIS, mở rộng được |

**Lời khuyên.** Học cách 3 trước (ngắn và có chứng minh gọn), dùng cách 2 làm oracle kiểm tra, biết cách 5 như công cụ LIS tổng quát.

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [38]:
print(Solution().findLongestChain([[1, 2], [2, 3], [3, 4]]))
print(Solution().findLongestChain([[1, 2], [7, 8], [4, 5]]))
print(Solution().findLongestChain([[5, 6]]))

2
3
1


**Bài 2 (tính tay).** Với `pairs = [[1,5],[2,3],[4,6],[7,8]]`, sắp theo mút phải, chạy tham lam bằng tay (ghi `end` sau mỗi lần chọn) và gán độ dài chuỗi tối đa vào `ans2` (đang là `None`).

In [39]:
ans2 = None      # TODO: một số nguyên

**Bài 3 (điền TODO).** Viết `is_chain(seq)` nhận một list các cặp **theo thứ tự đã chọn** và trả về `True` nếu **mỗi cặp đi sau cặp ngay trước nó** (nhỏ hơn hẳn). List rỗng hoặc một cặp thì `True`. Ví dụ `is_chain([[1,2],[4,5]]) -> True`, `is_chain([[1,2],[2,3]]) -> False`.

In [40]:
def is_chain(seq):
    # TODO: so sánh từng cặp liên tiếp: mút phải trước < mút trái sau
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_chain(pairs)` theo cách **tham lam theo mút phải** (cách 3).

In [41]:
def my_chain(pairs):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) `max_chain_with_gap(pairs, g)`: cặp sau phải **cách cặp trước ít nhất `g`**, nghĩa là `left_sau - right_trước >= g` (với `g = 1` chính là đề gốc). (b) `count_removals_for_disjoint(pairs)`: số cặp ít nhất phải **bỏ** để các cặp còn lại đôi một **không chung điểm** (không giao nhau, kể cả chạm đầu mút; nghĩa là đúng đề gốc: `n` trừ độ dài chuỗi dài nhất).

In [42]:
def max_chain_with_gap(pairs, g):
    # TODO
    pass

def count_removals_for_disjoint(pairs):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: nhớ điều kiện là `2 < 2` sai (chạm nhau không được).
- Bài 2: sắp theo mút phải: `[2,3], [1,5], [4,6], [7,8]`.
- Bài 3: lặp `i` từ `0` đến `len(seq) - 2`.
- Bài 4: `end = -inf`, sắp theo mút phải, `if a > end`.
- Bài 5: (a) thay điều kiện `a > end` bằng điều kiện có khoảng cách `g`; (b) dùng đáp án gốc.

**Mức 2 (vừa):**
- Bài 2: chọn `[2,3]` (`end = 3`); `[1,5]` bỏ (`1 <= 3`); `[4,6]` chọn (`end = 6`); `[7,8]` chọn.
- Bài 3: `all(seq[i][1] < seq[i + 1][0] for i in range(len(seq) - 1))`.
- Bài 4: `count += 1; end = b` khi chọn.
- Bài 5: (a) `a - end >= g` tức `a >= end + g` (với `end = -inf` thì luôn đúng; chú ý `-inf + g = -inf` vẫn ổn); (b) `len(pairs) - Solution().findLongestChain(pairs)`.

**Mức 3 (gần đáp án):**
- Bài 3: `return all(a[1] < b[0] for a, b in zip(seq, seq[1:]))`.
- Bài 4: xem mục 9, bỏ chú thích.
- Bài 5: (a) giữ nguyên tham lam theo mút phải; điều kiện chọn `a >= end + g`; chứng minh hoán đổi vẫn đúng vì mút phải nhỏ hơn thì vẫn "chừa" nhiều khoảng hơn.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [43]:
# Bài 1
assert Solution().findLongestChain([[1, 2], [2, 3], [3, 4]]) == 2
assert Solution().findLongestChain([[1, 2], [7, 8], [4, 5]]) == 3
assert Solution().findLongestChain([[5, 6]]) == 1

# Bài 2: [2,3] (end=3) -> bỏ [1,5] -> [4,6] (end=6) -> [7,8]  => 3
ans2 = 3
assert ans2 == Solution().findLongestChain([[1, 5], [2, 3], [4, 6], [7, 8]])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [44]:
# Bài 3
def is_chain(seq):
    return all(a[1] < b[0] for a, b in zip(seq, seq[1:]))

assert is_chain([[1, 2], [4, 5]]) is True
assert is_chain([[1, 2], [2, 3]]) is False
assert is_chain([]) is True
assert is_chain([[1, 2]]) is True
assert is_chain([[1, 2], [4, 5], [7, 8]]) is True
assert is_chain([[4, 5], [1, 2]]) is False
assert is_chain(longest_chain_path([[1, 2], [7, 8], [4, 5]])) is True
print("Bài 3 OK")

Bài 3 OK


In [45]:
# Bài 4
def my_chain(pairs):
    end, count = float("-inf"), 0
    for a, b in sorted(pairs, key=lambda p: p[1]):
        if a > end:
            count += 1
            end = b
    return count

rng = random.Random(4)
for _ in range(300):
    pairs = random_pairs(rng, rng.randint(1, 10), *rng.choice([(0, 8), (-5, 5), (-1000, 1000)]))
    assert my_chain(pairs) == oracle_subsets(pairs)
print("Bài 4 OK")

Bài 4 OK


In [46]:
# Bài 5 (a): cặp sau cách cặp trước ít nhất g
def max_chain_with_gap(pairs, g):
    end, count = float("-inf"), 0
    for a, b in sorted(pairs, key=lambda p: p[1]):
        if a - end >= g:
            count += 1
            end = b
    return count

def oracle_gap(pairs, g):
    n, best = len(pairs), 0
    for mask in range(1, 1 << n):
        chosen = sorted((pairs[i] for i in range(n) if mask >> i & 1), key=lambda p: (p[1], p[0]))
        if all(chosen[i + 1][0] - chosen[i][1] >= g for i in range(len(chosen) - 1)):
            best = max(best, len(chosen))
    return best

assert max_chain_with_gap([[1, 2], [2, 3], [3, 4]], 1) == 2          # g = 1 là đề gốc
assert max_chain_with_gap([[1, 2], [2, 3], [3, 4]], 2) == 1          # cần cách >= 2
assert max_chain_with_gap([[1, 2], [4, 5], [7, 8]], 2) == 3
assert max_chain_with_gap([[1, 2], [4, 5], [7, 8]], 3) == 2
assert max_chain_with_gap([[1, 2], [4, 5], [7, 8]], 4) == 2
assert max_chain_with_gap([[1, 2], [4, 5], [7, 8]], 6) == 1
rng = random.Random(55)
for _ in range(300):
    pairs = random_pairs(rng, rng.randint(1, 9), 0, 12)
    g = rng.randint(1, 4)
    assert max_chain_with_gap(pairs, g) == oracle_gap(pairs, g), (pairs, g)

# Bài 5 (b)
def count_removals_for_disjoint(pairs):
    return len(pairs) - Solution().findLongestChain(pairs)

assert count_removals_for_disjoint([[1, 2], [2, 3], [3, 4]]) == 1
assert count_removals_for_disjoint([[1, 2], [7, 8], [4, 5]]) == 0
assert count_removals_for_disjoint([[1, 5], [2, 3], [4, 6]]) == 1
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (tham lam theo mút phải):**

```python
class Solution:
    def findLongestChain(self, pairs: List[List[int]]) -> int:
        end = float("-inf")                          # mút phải của chuỗi hiện tại
        count = 0
        for a, b in sorted(pairs, key=lambda p: p[1]):   # sắp theo MÚT PHẢI
            if a > end:                              # nhỏ hơn HẲN
                count += 1
                end = b
        return count
```

**Template (DP kiểu LIS, O(n²)):**

```python
pairs.sort()
dp = [1] * len(pairs)
for i in range(len(pairs)):
    for j in range(i):
        if pairs[j][1] < pairs[i][0]:
            dp[i] = max(dp[i], dp[j] + 1)
return max(dp)
```

**Câu thần chú:** *khoảng không giao nhau nhiều nhất ⇒ sắp theo mút phải ⇒ chọn cái kết thúc sớm nhất ⇒ `a > end` (chặt).*

**Checklist trước khi submit:**

- [ ] Sắp theo **mút phải** (hoặc, nếu sắp theo mút trái, có co `end = min(end, b)`)?
- [ ] Điều kiện nối là `a > end` (chặt), **không** phải `>=`?
- [ ] `end` khởi tạo `float("-inf")` (không phải `0`) vì có số âm?
- [ ] Nếu làm DP: đã sắp xếp trước và trả `max(dp)` (không phải `dp[-1]`)?
- [ ] Không phụ thuộc thứ tự input (đề cho chọn thứ tự bất kỳ)?
- [ ] Đã thử: `n = 1`, cặp trùng nhau, cặp lồng nhau, cặp chạm nhau (`[1,2],[2,3]`), toàn số âm, biên `-1000` và `1000`?